# Feature Store and ML Baselines

Feature-set materialization, leakage guardrails, and baseline train/test shape.

In [1]:
from pathlib import Path
import json
import os
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    for parent in ROOT.parents:
        if (parent / "src").exists():
            ROOT = parent
            break

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

os.chdir(ROOT)
SAMPLE_ROWS = int(os.getenv("GTD_NOTEBOOK_SAMPLE_ROWS", "25000"))
print(f"Project root: {ROOT}")
print(f"Notebook sample rows: {SAMPLE_ROWS:,}")

Project root: C:\Users\kunmi\Personal Projects\Global-Terrorism-Database
Notebook sample rows: 25,000


In [2]:
from sklearn.model_selection import train_test_split

from gtd_capstone.data.repository import DataRepository
from gtd_capstone.features.store import materialize_feature_set
from gtd_capstone.ml.experiment_suite import FEATURES

incidents = DataRepository().load_incidents(sample_rows=min(SAMPLE_ROWS, 10000))
features = materialize_feature_set(incidents)
x = features[FEATURES]
y = features["severity"]
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y if y.value_counts().min() >= 2 else None,
)
{
    "feature_set": features.attrs["feature_set"],
    "leakage_blocklist": features.attrs["leakage_blocklist"],
    "train_rows": len(x_train),
    "test_rows": len(x_test),
    "features": FEATURES,
}

{'feature_set': 'incident_core_v1',
 'leakage_blocklist': ['casualties', 'nkill', 'nwound', 'severity'],
 'train_rows': 7500,
 'test_rows': 2500,
 'features': ['iyear',
  'imonth',
  'suicide',
  'property',
  'ishostkid',
  'region_txt',
  'country_txt',
  'attacktype1_txt',
  'targtype1_txt',
  'weaptype1_txt']}

In [3]:
y.value_counts().rename_axis("severity").reset_index(name="rows")

,severity,rows
0,None,6082
1,Low,3185
2,Medium,384
3,High,264
4,Mass Casualty,85
